# Ingesta de datos de la disponibilidad de Acuíferos en Sonora

Esta libreta de Jupyter se usará para descargar, limpiar y organizar los datos oficiales de la CONAGUA sobre los acuíferos de Sonora. A partir de variables como la recarga, la extracción y el nivel de sobreexplotación, analizaremos la presión real sobre el agua para entender qué tan vulnerables están las distintas zonas del estado.

## Ingesta de datos

### Librerías necesarias

In [1]:
#!pip install -q pandas requests pydantic

In [2]:
import io
from pathlib import Path
import pandas as pd
import requests
from pydantic import BaseModel, ValidationError, Field, ConfigDict, field_validator
from typing import Optional
from loguru import logger

root = Path().resolve().parent  
raw_dir = root / "data/raw/"
raw_dir.mkdir(parents=True, exist_ok=True)

output_csv = raw_dir / "acuiferos_sonora.csv"

url = "https://sigagis.conagua.gob.mx/gas1/sections/Edos/sonora/sonora.html"

# Headers to simulate a web browser and avoid blocking/redirection
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

## Descarga de datos

In [3]:
try:
  logger.info("Descargando página de CONAGUA...")
  response = requests.get(url, headers=headers)
  response.raise_for_status()  # Verifica que la petición sea exitosa (código 200)

  # Use io.StringIO to read the HTML content downloaded with requests
  tables = pd.read_html(io.StringIO(response.text))

  if tables:
    # We select the main aquifer table
    df_aquifers = tables[0]

    logger.success(
        f"¡Éxito! Se encontraron {len(tables)} tablas. Mostrando las primeras"
        " filas:"
    )
    display(df_aquifers.head())

    # Save as CSV
    df_aquifers.to_csv(
        output_csv, index=False, encoding="utf-8-sig"
    )
    logger.success("\nArchivo guardado exitosamente como 'acuiferos_sonora.csv'")
  else:
    logger.warning("No se encontraron tablas HTML en la página.")

except Exception as e:
  logger.warning(f"Ocurrió un error al procesar la solicitud: {e}")

2026-09-28 20:37:50.399 | INFO     | __main__:<module>:2 - Descargando página de CONAGUA...
2026-09-28 20:37:51.048 | SUCCESS  | __main__:<module>:13 - ¡Éxito! Se encontraron 3 tablas. Mostrando las primeras filas:


,CLAVE,ACUÃFERO,R,DNC,VEAS,DMA,DOCUMENTO
0,2601,VALLE DE SAN LUIS RÃO COLORADO,236.8,32.5,292.228164,-87.928164,DR_2601
1,2602,LOS VIDRIOS,6.7,2.0,3.666326,1.033674,DR_2602
2,2603,SONOYTA-PUERTO PEÃASCO,41.4,15.9,109.216413,-83.716413,DR_2603
3,2604,ARROYO SAHUARO,10.9,3.3,9.420099,-1.820099,DR_2604
4,2605,CABORCA,197.8,0.0,334.320085,-136.520085,DR_2605


2026-09-28 20:37:51.064 | SUCCESS  | __main__:<module>:23 - 
Archivo guardado exitosamente como 'acuiferos_sonora.csv'


## Validación de la descarga de datos

### Schema basandonos en el diccionario de datos

Nota: En este esquema DMA (Disponibilidad Media Anual) no lleva la restricción ge=0, ya que puede tomar valores negativos cuando el acuífero se encuentra en déficit o sobreexplotación

In [ ]:
class AcuiferoSonoraSchema(BaseModel):
  model_config = ConfigDict(extra="ignore")

  CLAVE: Optional[str] = None
  ACUIFERO: Optional[str] = None
  R: Optional[float] = None
  DNC: Optional[float] = None
  VEAS: Optional[float] = None
  DMA: Optional[float] = None
  DOCUMENTO: Optional[str] = None

  @field_validator("*", mode="before")
  @classmethod
  def normalize_missing_values(cls, value):
    # If it is None, an empty string, or pandas NaN, convert it to None
    if value is None or pd.isna(value) or str(value).strip() == "":
      return None
    return value

  @field_validator("CLAVE", "ACUIFERO", "DOCUMENTO", mode="before")
  @classmethod
  def convert_text_fields(cls, value):
    if value is None or pd.isna(value):
      return None
    return str(value).strip()

  @field_validator("R", "DNC", "VEAS", "DMA", mode="before")
  @classmethod
  def convert_numeric_fields(cls, value):
    if value is None or pd.isna(value) or str(value).strip() == "":
      return None
    try:
      return float(value)
    except (ValueError, TypeError):
      return None

### Función para validar los datos descargados

In [5]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            agriculture = AcuiferoSonoraSchema(**row.to_dict())
            validated_rows.append(agriculture.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [6]:
df_validated = validate_df(df_aquifers)
df_validated.head(5)

,CLAVE,ACUIFERO,R,DNC,VEAS,DMA,DOCUMENTO
0,2601,None,236.8,32.5,292.228164,-87.928164,DR_2601
1,2602,None,6.7,2.0,3.666326,1.033674,DR_2602
2,2603,None,41.4,15.9,109.216413,-83.716413,DR_2603
3,2604,None,10.9,3.3,9.420099,-1.820099,DR_2604
4,2605,None,197.8,0.0,334.320085,-136.520085,DR_2605
